In [2]:
from build_context import stringify_report, format_triplet, construct_query_context
from config import *
from langchain_community.graphs import Neo4jGraph
from langchain_community.embeddings import HuggingFaceEmbeddings
from neo4j import GraphDatabase, Result
from typing import Dict, Any
import torch
import pandas as pd
import helpers

# KG-RAG setup

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD),database=NEO4J_DATABASE)

def db_query(cypher: str, params: Dict[str, Any] = {}) -> pd.DataFrame:
    """Executes a Cypher statement and returns a DataFrame"""
    return driver.execute_query(
        cypher, parameters_=params, result_transformer_=Result.to_df
    )

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    refresh_schema=False,
    driver_config={"notifications_disabled_classifications": ["DEPRECATION"]}
)

embedding = HuggingFaceEmbeddings(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

/home/damon/kg_aug_causal_disc_exp/build_context.py:12: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(NEO4J_URI,NEO4J_USERNAME,NEO4J_PASSWORD,NEO4J_DATABASE, refresh_schema=False)
/shared/graphrag/lib/python3.10/site-packages/langchain_community/graphs/neo4j_graph.py:404: PreviewWarning: notifications_disabled_classifications is a preview feature. It might be changed without following the deprecation policy. See also https://github.com/neo4j/neo4j-python-driver/wiki/preview-features.
  self._driver = neo4j.GraphDatabase.driver(
/tmp/ipykernel_895029/2962234265.py:30: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be

In [9]:
# First, let's check what relationship types exist
rel_types_query = """
MATCH (n:__Entity__)-[r]->(m:__Entity__)
RETURN DISTINCT type(r) as rel_type, count(r) as count
ORDER BY count DESC
"""

rel_types_df = db_query(rel_types_query)
print("Relationship types in the graph:")
rel_types_df.head(60)

Relationship types in the graph:


,rel_type,count
0,ASSOCIATION,4125
1,ASSOCIATED_WITH,1399
2,INFLUENCES,892
3,AFFECTS,784
4,RISK_FACTOR,589
5,CAUSES,546
6,CO-AUTHOR,492
7,INFLUENCE,311
8,RELATED_TO,301
9,HAS,290
